# See stimuli from the original dataset 

In [1]:
import altair as alt 
import pandas as pd 
import polars as pl

## Experiment 1

- **Only looked at line charts** 
- “We used a 2 (variability upper vs. lower `flip`) × 2 (more vs. less variability `noise`?) within-subjects design for a total of 4 stimuli types of interest.”
- Used 12 seeds $\times$ 4 stimuli -> 48 stimuli data set
- All participant was shown all 48 images 

In [39]:
df_1 = pl.read_json("../../data/moritz/stimuli/exp1-stimuli.json")
df_1.head(5)

# # pandas way 
# df_1 = pd.read_json("../../data/moritz/stimuli/exp1-stimuli.json")
# df_1.head(5)

index,seed,noise,flip,type,data,lineData,highNoiseData
i64,i64,list[f64],bool,str,list[struct[2]],list[struct[2]],list[struct[2]]
0,1,"[0.0, 0.15]",false,"""line""","[{0,0.836738}, {1,0.846661}, … {119,0.0}]","[{0,0.836738}, {1,0.846661}, … {119,0.0}]","[{0,0.812146}, {1,0.872021}, … {119,0.0}]"
1,1,"[0.0, 0.15]",true,"""line""","[{0,0.163262}, {1,0.153339}, … {119,1.0}]","[{0,0.163262}, {1,0.153339}, … {119,1.0}]","[{0,0.187854}, {1,0.127979}, … {119,1.0}]"
2,1,"[0.0, 0.4]",false,"""line""","[{0,0.812146}, {1,0.872021}, … {119,0.0}]","[{0,0.812146}, {1,0.872021}, … {119,0.0}]","[{0,0.812146}, {1,0.872021}, … {119,0.0}]"
3,1,"[0.0, 0.4]",true,"""line""","[{0,0.187854}, {1,0.127979}, … {119,1.0}]","[{0,0.187854}, {1,0.127979}, … {119,1.0}]","[{0,0.187854}, {1,0.127979}, … {119,1.0}]"
4,2,"[0.0, 0.15]",false,"""line""","[{0,0.817333}, {1,0.800974}, … {119,0.023628}]","[{0,0.817333}, {1,0.800974}, … {119,0.023628}]","[{0,0.871047}, {1,0.826411}, … {119,0.02579}]"


In [51]:
# 4 stimuli type of interest 
df_1.group_by(["noise", "flip"]).len()

noise,flip,len
list[f64],bool,u32
"[0.0, 0.4]",true,12
"[0.0, 0.4]",false,12
"[0.0, 0.15]",false,12
"[0.0, 0.15]",true,12


The three columns, `data`, `lineData` and `highNoiseData` is quite confusing. Here's how to understand them: 

- When `noise == [0.0, 0.4]`, i.e., high noise, `data == lineData == highNoiseData`
- When `noise == [0.0, 0.15]`, i.e., low noise, `data = lineData != highNoiseData`

So to recreate Experiment 1 from Mortiz, one only needs data from `data` or `lineData`. 

- [ ] The next step is to figure out programmatically how to do the Vega-Embed + slider + new experiment thingy with ReVISit ... 

Define custom plotting data for row with index `i`: 

In [54]:
# define a function that generates things 
from typing import Literal 

SimulationType = ["data", "lineData", "highNoiseData"]
PlotType = ["point", "line"]

def get_plot_df(index: int):
    return df_1.filter(pl.col("index") == index).select("data").explode("data").unnest("data")

def plot(index: int, plot_type: PlotType, col: SimulationType): 
    plot_df = df_1.filter(pl.col("index") == index).select(col).explode(col).unnest(col)
    if plot_type == "point": 
        p = alt.Chart(plot_df).mark_point(filled=True).encode(
            x=alt.X("x", axis=None),
            y=alt.Y("y", axis=None).scale(domain=[0, 1])
        ).properties(
            width=500,
            height=200
        )
    elif plot_type == "line":
        p = alt.Chart(plot_df).mark_line().encode(
            x=alt.X("x", axis=None),
            y=alt.Y("y", axis=None).scale(domain=[0, 1])
        ).properties(
            width=500,
            height=200
        )
    return p

In [65]:
# try a strip plot to see distribution of y values 

alt.Chart(get_plot_df(0)).mark_tick().encode(x = "y")

alt.Chart(...)

In [67]:
# try a strip plot to see y distribution 

alt.Chart(get_plot_df(5)).mark_tick().encode(x = "y")

alt.Chart(...)

In [73]:
df_1_p = df_1.select(["index", "seed", "noise", "flip", "data" ]).with_columns(
    pl.col("data").list.eval(pl.element().struct.field("y")).list.mean().alias("mean_y")
)
df_1_p

index,seed,noise,flip,data,mean_y
i64,i64,list[f64],bool,list[struct[2]],f64
0,1,"[0.0, 0.15]",false,"[{0,0.836738}, {1,0.846661}, … {119,0.0}]",0.522147
1,1,"[0.0, 0.15]",true,"[{0,0.163262}, {1,0.153339}, … {119,1.0}]",0.477853
2,1,"[0.0, 0.4]",false,"[{0,0.812146}, {1,0.872021}, … {119,0.0}]",0.522147
3,1,"[0.0, 0.4]",true,"[{0,0.187854}, {1,0.127979}, … {119,1.0}]",0.477853
4,2,"[0.0, 0.15]",false,"[{0,0.817333}, {1,0.800974}, … {119,0.023628}]",0.420407
…,…,…,…,…,…
43,11,"[0.0, 0.4]",true,"[{0,0.814328}, {1,0.73583}, … {119,1.0}]",0.445752
44,19,"[0.0, 0.15]",false,"[{0,0.920993}, {1,0.873257}, … {119,0.188542}]",0.256614
45,19,"[0.0, 0.15]",true,"[{0,0.079007}, {1,0.126743}, … {119,0.811458}]",0.743386


In [79]:
alt.Chart(df_1_p.select("mean_y")).transform_density(
    'mean_y',
    as_=['mean_y', 'density'],
).mark_area().encode(
    x=alt.X("mean_y").scale(domain=[0, 1]),
    y='density:Q',
)

alt.Chart(...)

In [70]:
plot(0, "line", "data")

alt.Chart(...)

In [41]:
plot(0, "line", "lineData")

alt.Chart(...)

In [42]:
plot(0, "line", "highNoiseData")

alt.Chart(...)

In [43]:
plot(1, "line", "data")

alt.Chart(...)

In [45]:
plot(1, "line", "lineData")

alt.Chart(...)

In [46]:
plot(1, "line", "highNoiseData")

alt.Chart(...)

In [48]:
plot(2, "line", "data")

alt.Chart(...)

In [49]:
plot(2, "line", "lineData")

alt.Chart(...)

In [50]:
plot(2, "line", "highNoiseData")

alt.Chart(...)

In [17]:
plot(1, "point", "highNoiseData")

alt.Chart(...)

Somehow this is making me think about gradience and the judgment of variance / mean ... 

## Experiment 2 

- “but we encoded the data using 1) Cartesian spaced points, 2) points equally spaced along the arc of the line, and 3) the same line encoding used in Experiment 1 (see Figure 4)”
- “We used a 3 (point along x, point along arc, line) × 2 (variability upper vs. lower) × 2 (more variability vs. no variability) design.”
- 12 seeds
- each participant viewed 48 images from one plot type 

In [66]:
df_2 = pl.read_json("../../data/moritz/stimuli/exp2-stimuli.json")
# df_2.head(5)

In [30]:
df_2.group_by(["noise", "flip", "type"]).len()

noise,flip,type,len
list[f64],bool,str,u32
"[0.0, 0.4]",false,"""line""",12
"[0.0, 0.0]",true,"""line""",12
"[0.0, 0.0]",false,"""line""",12
"[0.0, 0.0]",true,"""point_arc""",12
"[0.0, 0.0]",false,"""point_arc""",12
…,…,…,…
"[0.0, 0.4]",true,"""line""",12
"[0.0, 0.0]",false,"""point""",12
"[0.0, 0.4]",true,"""point_arc""",12
